# Digit Recognition

You can create very useful clusters without having labeled data. Let's test with MNIST

0. Import usuals librairies

In [1]:
#!pip install -q xgboost
#!pip install -q s3fs
 #!pip install -U kaleido

# Load in our libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.figure_factory as ff

# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "colab" # pour que colab ne bloque pas l'export svg

import warnings
warnings.filterwarnings('ignore')

# datasets
from sklearn.datasets import load_sample_image, load_digits

# pipeline transfomers
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# split
from sklearn.model_selection import train_test_split, GridSearchCV

# regression
from sklearn.linear_model import LogisticRegression, Ridge

# arbres de décision
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

# import ensemble methods
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, AdaBoostRegressor, GradientBoostingClassifier, VotingRegressor, StackingClassifier,  StackingRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor, XGBClassifier

# metrics
from sklearn.metrics import r2_score, accuracy_score, silhouette_score, confusion_matrix

### ML non supervisé
from sklearn.cluster import KMeans, MiniBatchKMeans

# montage drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


1. In Scikit Learn, import _load_digits_

In [2]:
digits = load_digits()
digits

{'data': array([[ 0.,  0.,  5., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ..., 10.,  0.,  0.],
        [ 0.,  0.,  0., ..., 16.,  9.,  0.],
        ...,
        [ 0.,  0.,  1., ...,  6.,  0.,  0.],
        [ 0.,  0.,  2., ..., 12.,  0.,  0.],
        [ 0.,  0., 10., ..., 12.,  1.,  0.]]),
 'target': array([0, 1, 2, ..., 8, 9, 8]),
 'frame': None,
 'feature_names': ['pixel_0_0',
  'pixel_0_1',
  'pixel_0_2',
  'pixel_0_3',
  'pixel_0_4',
  'pixel_0_5',
  'pixel_0_6',
  'pixel_0_7',
  'pixel_1_0',
  'pixel_1_1',
  'pixel_1_2',
  'pixel_1_3',
  'pixel_1_4',
  'pixel_1_5',
  'pixel_1_6',
  'pixel_1_7',
  'pixel_2_0',
  'pixel_2_1',
  'pixel_2_2',
  'pixel_2_3',
  'pixel_2_4',
  'pixel_2_5',
  'pixel_2_6',
  'pixel_2_7',
  'pixel_3_0',
  'pixel_3_1',
  'pixel_3_2',
  'pixel_3_3',
  'pixel_3_4',
  'pixel_3_5',
  'pixel_3_6',
  'pixel_3_7',
  'pixel_4_0',
  'pixel_4_1',
  'pixel_4_2',
  'pixel_4_3',
  'pixel_4_4',
  'pixel_4_5',
  'pixel_4_6',
  'pixel_4_7',
  'pixel_5_0',
  'pixel_5_1',
 

2. Look at the [Load_digit](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html) documentation and store the numbers in a _numbers_ variable and your target variable in a _target_ variable.

In [3]:
help(load_digits)


Help on function load_digits in module sklearn.datasets._base:

load_digits(*, n_class=10, return_X_y=False, as_frame=False)
    Load and return the digits dataset (classification).

    Each datapoint is a 8x8 image of a digit.

    =================   ==============
    Classes                         10
    Samples per class             ~180
    Samples total                 1797
    Dimensionality                  64
    Features             integers 0-16
    =================   ==============

    This is a copy of the test set of the UCI ML hand-written digits datasets
    https://archive.ics.uci.edu/ml/datasets/Optical+Recognition+of+Handwritten+Digits

    Read more in the :ref:`User Guide <digits_dataset>`.

    Parameters
    ----------
    n_class : int, default=10
        The number of classes to return. Between 0 and 10.

    return_X_y : bool, default=False
        If True, returns ``(data, target)`` instead of a Bunch object.
        See below for more information about 

In [4]:
numbers = digits.data      # tableau (1797, 64)
numbers

array([[ 0.,  0.,  5., ...,  0.,  0.,  0.],
       [ 0.,  0.,  0., ..., 10.,  0.,  0.],
       [ 0.,  0.,  0., ..., 16.,  9.,  0.],
       ...,
       [ 0.,  0.,  1., ...,  6.,  0.,  0.],
       [ 0.,  0.,  2., ..., 12.,  0.,  0.],
       [ 0.,  0., 10., ..., 12.,  1.,  0.]])

In [5]:
target = digits.target     # labels réels (0–9)
target

array([0, 1, 2, ..., 8, 9, 8])

3. Tentons de visualiser quelques nombres. Regardez d’abord la taille de votre dataset. Combien de colonnes voyez vous ?

Devinez ensuite quelle taille d’image ce nombre de colonnes devraient donner

In [6]:
numbers.shape
print (f"nombre d’images : {numbers.shape[0]}")
print (f"nombre de colonnes {numbers.shape[1]}")
print(f"=> chaque image est représentée par 64 pixels")

nombre d’images : 1797
nombre de colonnes 64
=> chaque image est représentée par 64 pixels


 images: {ndarray} of shape (1797, 8, 8)
            The raw image data.
        DESCR: str
            The full description of the dataset.

    (data, target) : tuple if ``return_X_y`` is True
        A tuple of two ndarrays by default. The first contains a 2D ndarray of
        shape (1797, 64) with each row representing one sample and each column
        representing the features.
        

4. Now look at the documentation related to [imshow](https://plotly.com/python/imshow/) from plotly. Try to view a random number. Add as a title, the number this image corresponds to. Then try to view 10 random numbers in the dataset

In [7]:
# Sélection d'un index aléatoire
idx = np.random.randint(0, len(numbers))

# Image reshaped pour affichage
img = numbers[idx].reshape(8, 8)

# Affichage Plotly
fig = px.imshow(img, color_continuous_scale="gray")
fig.update_layout(title=f"Label : {target[idx]}")
fig.update_coloraxes(showscale=False)
fig.show()

In [8]:
# Créer une figure 2 lignes × 5 colonnes
fig = make_subplots(rows=2, cols=5, subplot_titles=[f"Idx {i}" for i in range(10)])

for i in range(10):
    idx = np.random.randint(0, len(numbers))
    img = numbers[idx].reshape(8, 8)

    fig.add_trace(
        go.Heatmap(
            z=img,
            colorscale="gray",
            showscale=False
        ),
        row=1 if i < 5 else 2,
        col=(i % 5) + 1
    )

    # Donner comme titre le vrai label
    fig.layout.annotations[i].text = f"Label : {target[idx]}"

fig.update_layout(height=600, width=900, title="10 images aléatoires du dataset MNIST (load_digits)")
fig.show()


5. We're going to apply the KMeans to our dataset, how many clusters do you think we're going to initialize the algorithm on?

---> 10 because we have 10 numbers in the dataset!
évident : on a 10 classes => 10 clusters

6. Create your KMeans algorithm with the right number of clusters

In [9]:
kmeans = KMeans(
    n_clusters=10,
    random_state=42,
    n_init=10  # The default value of `n_init` will change from 10 to 'auto'
    # in 1.4. Set the value of `n_init` explicitly to suppress the warning
)

# pas de fit_predict ici
kmeans.fit(numbers)
# d'abord fit pour récupérer l'objet entrainé
preds = kmeans.predict(numbers)
# remapper si besoin vers les vraies classes


7. Let's evaluate our model, calculate the _accuracy_score_ of our predictions by importing the sklearn module. What do you conclude?

In [10]:
acc = accuracy_score(target, preds)
print("Accuracy:", acc)

Accuracy: 0.03951029493600445


KMeans est un algorithme non supervisé.

Cela implique :

Il ne connaît pas les vraies étiquettes (0 à 9).

Le numéro des clusters (0, 1, 2, …, 9) est complètement arbitraire.

Le cluster 0 ne représente pas forcément le chiffre 0.

Le cluster 1 peut très bien représenter le chiffre 7, etc.

Résultat :

→ Même si les images sont bien regroupées,
→ la comparaison brute avec target n’a AUCUN sens.

C’est donc normal que l’accuracy soit autour de 4 %.

8. Look at the coordinates of the centroids (cf. [cluster_center_](http://lijiancheng0614.github.io/scikit-learn/modules/generated/sklearn.cluster.KMeans.html))

In [11]:
centroids = kmeans.cluster_centers_
print (f"nombre d’images : {centroids.shape[0]}")
print (f"nombre de colonnes {centroids.shape[1]}")
print(f"=> chaque image est représentée par 64 pixels")


nombre d’images : 10
nombre de colonnes 64
=> chaque image est représentée par 64 pixels


9. Try to visualize each of the centroids and compare them with the different labels. What do you notice?

In [12]:
fig = make_subplots(rows=2, cols=5, subplot_titles=[f"Cluster {i}" for i in range(10)])

for i in range(10):
    fig.add_trace(
        go.Heatmap(
            z=kmeans.cluster_centers_[i].reshape(8, 8),
            colorscale='gray',
            showscale=False
        ),
        row=1 if i < 5 else 2,
        col=(i % 5) + 1
    )

fig.update_layout(height=600, width=900, title="Centroïdes KMeans")
fig.show()

----> Looks like good predictions! It just seems that centroids don't necessarily match the label.

Certains centroïdes sont flous ou ambigus

Particulièrement :

3, 5, 8 → formes similaires → centroïdes difficiles à interpréter

4 et 9 → erreurs fréquentes, centroïdes très proches

7 et 1 → si certains “7” sont écrits sans la barre horizontale, KMeans les mélange

Ces flous viennent du fait que KMeans fait une moyenne pixel par pixel, ce qui crée des formes mixtes.

10. We will try to match our cluster labels with the target values. Here are some clues:

    a. Identify the most frequent target value for observations in cluster 1.

In [13]:
# Les prédictions brutes de KMeans
preds = kmeans.labels_   # équivalent à kmeans.predict(numbers)

# Créer un tableau associant prédiction et vraie étiquette
df = pd.DataFrame({
    "cluster": preds,
    "target": target
})

# Compter les vraies classes dans le cluster 1
cluster_1_counts = df[df["cluster"] == 1]["target"].value_counts()
# df[df["cluster"] == 1] applique ce filtre au df en ne gardant que les observations appartenant au cluster 1
# puis renvoie une série pandas avec les valeurs de la colonne target (première valeur 0)
# ensuite compte combien de fois chaque chiffre apparait dans ce cluster (177 fois)
# in fine le cluster 1 correspond au chiffre mnist 0
print(cluster_1_counts)

target
0    177
6      1
2      1
Name: count, dtype: int64


b. Programming a loop which allows to create a label vector which contains for each observation the target value corresponding to the cluster to which it belongs.

In [14]:
mapping = {} # un dictionnaire en mode clé valeur

for c in range(10):  # 10 clusters
    # valeurs cibles dans ce cluster
    cluster_targets = target[preds == c]

    # classe majoritaire
    most_common = np.bincount(cluster_targets).argmax()

    mapping[c] = most_common

print("Mapping cluster → vraie étiquette :")
print(mapping)


Mapping cluster → vraie étiquette :
{0: np.int64(2), 1: np.int64(0), 2: np.int64(1), 3: np.int64(8), 4: np.int64(7), 5: np.int64(6), 6: np.int64(3), 7: np.int64(5), 8: np.int64(9), 9: np.int64(4)}


In [15]:
mapped_preds = np.array([mapping[c] for c in preds])
mapped_preds[:20]  # premières valeurs remappées



array([0, 8, 8, 3, 4, 9, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [16]:
target[:20]

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

11. Re-evaluate your model. What is your new *accuracy_score*?

In [17]:
accuracy_remapped = accuracy_score(target, mapped_preds)
accuracy_remapped

0.7935447968836951

12. Look at the numbers where our algorithm got it wrong the most via a confusion matrix...

In [18]:
cm = confusion_matrix(target, mapped_preds)

fig = ff.create_annotated_heatmap(
    z=cm,
    x=[str(i) for i in range(10)],  # prédictions
    y=[str(i) for i in range(10)],  # vraies classes
    colorscale="Viridis"
)

fig.update_layout(
    title="Matrice de confusion – KMeans (labels remappés)",
    xaxis_title="Prédiction",
    yaxis_title="Vérité "
)

fig.show()